In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.preprocessing import LabelEncoder

# Find the dataset folder from the project root or this notebook folder
dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# a. Load and preprocess data
train = pd.read_csv(dataset_root / "FNN and CNN/Sign Language MNIST/sign_mnist_train.csv", dtype="uint8")
print("Training CSV shape:", train.shape)
print("Training preview (3 rows, 8 columns):\n", train.iloc[:3, :8])
print("Training label counts:\n", train["label"].value_counts().sort_index())
test = pd.read_csv(dataset_root / "FNN and CNN/Sign Language MNIST/sign_mnist_test.csv", dtype="uint8")
print("Test CSV shape:", test.shape)
print("Test preview (3 rows, 8 columns):\n", test.iloc[:3, :8])
print("Test label counts:\n", test["label"].value_counts().sort_index())


In [ ]:
# Prepare features and labels
y_train = train.pop("label")
y_test = test.pop("label")

x_train = train.to_numpy("float32").reshape(-1, 28, 28, 1) / 255
x_test = test.to_numpy("float32").reshape(-1, 28, 28, 1) / 255

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

print("Training features:", x_train.shape, "Training labels:", y_train.shape)
print("Test features:", x_test.shape, "Test labels:", y_test.shape)
print("First 10 training labels:", pd.Series(y_train).head(10).tolist())
print("Scaled training pixel range:", np.asarray(x_train).min(), np.asarray(x_train).max())
print("First image, first 10 scaled pixels:", np.asarray(x_train)[0].reshape(-1)[:10])
print("Label encoding (original label -> model label):")
print(pd.DataFrame({
    "Original label": encoder.classes_,
    "Model label": range(len(encoder.classes_))
}))


In [ ]:
# b. Define CNN architecture
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(24, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Network architecture:")
model.summary()
print("Optimizer:", type(model.optimizer).__name__)
print("Loss function:", model.loss)


In [ ]:
# c. Train model
history = model.fit(x_train, y_train, epochs=5, batch_size=128)
print("Final training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# d. Evaluate performance
loss, accuracy = model.evaluate(x_test, y_test)
print("Test loss:", loss)
print("Accuracy:", accuracy * 100, "%")

# Inspect five predictions without retraining
probabilities = model.predict(np.asarray(x_test)[:5], verbose=0)
predicted = probabilities.argmax(axis=1)
actual = np.asarray(y_test)[:5]
print("First test image class probabilities:", probabilities[0].round(3))
actual = encoder.inverse_transform(actual)
predicted = encoder.inverse_transform(predicted)
print("First five predictions:")
print(pd.DataFrame({"True label": actual, "Predicted label": predicted}))
